# CSAI3121 - Machine Learning and Intelligent Data Analysis
## Lab 2 - Simple and Multiple Linear Regression

<i>Based on the exercise from "Machine Learning with PyTorch and Scikit-Learn" book by Sebastian Raschka.</i>

In this 1-hour lab, you will use a small subset of the Ames Housing dataset to fit and compare two regression models for predicting `SalePrice`.

**Learning goals**
1. Load and prepare a real tabular dataset.
2. Use exploratory plots and correlations to choose useful predictors.
3. Fit a simple linear regression (SLR) model with one predictor.
4. Fit a multiple linear regression (MLR) model with several predictors.
5. Compare models using R^2, MSE, RMSE, and diagnostic plots.

The features we will use are:
- `Overall Qual`: overall material and finish rating from 1 (very poor) to 10 (excellent)
- `Overall Cond`: overall condition rating from 1 (very poor) to 10 (excellent)
- `Gr Liv Area`: above-ground living area in square feet
- `Central Air`: central air conditioning (N = no, Y = yes)
- `Total Bsmt SF`: total basement area in square feet
- `SalePrice`: sale price in U.S. dollars ($)

Work through the TODOs and answer the short reflection questions as you go.

---

### Step 1. Data Preprocessing (about 10 minutes)
In this section, you will:
- read the dataset
- inspect the first few rows and dimensions
- encode the categorical `Central Air` column
- check for missing values
- remove rows with missing values

In [ ]:
import pandas as pd

In [ ]:
# Read the data.
# We only load the columns needed for this lab.

columns = ['Overall Qual', 'Overall Cond', 'Gr Liv Area', 'Central Air', 'Total Bsmt SF', 'SalePrice']

df = pd.read_csv('AmesHousing.csv', sep='	', usecols=columns)

# Display summary information about the dataset.
df.info()

In [ ]:
# Display some first few rows of the data

df.head()

In [ ]:
# Display the dimensions of the data: (number of rows, number of columns)

df.shape

In [ ]:
# Convert the string values in Central Air into numeric values.
# N means no central air, Y means yes central air.

df['Central Air'] = df['Central Air'].map({'N': 0, 'Y': 1})
df.head()

In [ ]:
# Check whether each column has missing values.
df.isnull().sum()

In [ ]:
# Remove rows that contain missing values.
# axis=0 means rows are removed.

df = df.dropna(axis=0)
df.isnull().sum()

In [ ]:
# Check the new dimensions after removing missing values.
df.shape

### Step 2. Data Visualization (about 15 minutes)
Exploratory data analysis (EDA) helps us understand the data before fitting a model.

In this section, you will:
- inspect pairwise relationships between variables
- compute correlations between all numeric variables
- identify candidate predictors for `SalePrice`

If time is short, focus first on the correlation heatmap.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("darkgrid")

In [ ]:
# Optional visual overview: pairwise scatterplots and histograms.
# This can take a few seconds to render.

plt.figure(figsize=(8, 6))
g = sns.PairGrid(df, diag_sharey=False, corner=True, height=1.5, aspect=1.3)
g.map_diag(sns.histplot, bins=10, edgecolor='white')
g.map_lower(sns.scatterplot, s=20, edgecolor='black', alpha=0.4)
plt.show()

<div style="background-color: #f0f8ff; padding: 10px; border-radius: 5px;">
  <h3 style="color: #0056b3;">Quick Check 1: EDA</h3>
  <p>Answer briefly before moving on:</p>
  <ol>
    <li>Which variable seems most strongly related to `SalePrice`?</li>
    <li>Do you notice any possible outliers?</li>
    <li>Are any predictors strongly correlated with each other?</li>
  </ol>
</div>

In [ ]:
# Compute the correlation matrix.
# Positive values close to 1 indicate strong positive correlation.
# Negative values close to -1 indicate strong negative correlation.

cm = df.corr()
cm

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

# Other available colormaps: rocket, mako, viridis, plasma, coolwarm

#### Think Before Modeling
Based on the correlation heatmap, choose one predictor for the simple linear regression model.

For this lab, we will start with `Gr Liv Area` because it is easy to interpret and usually has a clear relationship with `SalePrice`.

### Step 3. Simple Linear Regression (SLR) (about 15 minutes)
We will first fit a model with one predictor:

`SalePrice = intercept + slope * Gr Liv Area`

`statsmodels` is useful here because it gives a detailed regression summary, including coefficients, R^2 values, and p-values.

In [ ]:
import statsmodels.api as sm

x = df[['Gr Liv Area']]
y = df['SalePrice']

# Add the intercept term.
x1 = sm.add_constant(x)

# Create and fit the OLS regression model.
olsm = sm.OLS(y, x1)
model = olsm.fit()

# View the fitted coefficients.
model.params

In [ ]:
# A more comprehensive analysis result can be found in the summary.
model.summary()

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error

# Generate the predicted SalePrice values.
y_pred = model.predict(x1)

plt.figure(figsize=(4, 4))
plt.scatter(x, y, edgecolor='black', alpha=0.5)
plt.plot(x, y_pred, color='red')
plt.xlabel('Living area above ground')
plt.ylabel('Sale price')
plt.title('SLR: SalePrice vs Gr Liv Area')
plt.show()

In [ ]:
# Compute SLR evaluation metrics.

slr_r2 = r2_score(y, y_pred)
slr_mse = mean_squared_error(y, y_pred)
slr_rmse = slr_mse ** 0.5

print(f'SLR R^2: {slr_r2:.4f}')
print(f'SLR MSE: {slr_mse:.2f}')
print(f'SLR RMSE: ${slr_rmse:,.2f}')

<div style="background-color: #f0f8ff; padding: 10px; border-radius: 5px;">
  <h3 style="color: #0056b3;">Quick Check 2: Interpret the SLR Model</h3>
  <p>Answer briefly before moving on:</p>
  <ol>
    <li>What does the slope for <code>Gr Liv Area</code> mean in words?</li>
    <li>Does the model explain all variation in house prices? How do you know?</li>
    <li>Is RMSE easier to interpret than MSE? Why?</li>
  </ol>
</div>


### Step 4. Multiple Linear Regression (MLR) (about 15 minutes)
Now fit a model using all available predictors except `SalePrice`.

Your task: modify the SLR pattern above to build the MLR model, print the summary, and compare the result with the SLR model.

In [ ]:
multi_x = df.drop(columns=['SalePrice'])
y = df['SalePrice']

multi_x1 = sm.add_constant(multi_x)

# TODO 1: create an OLS model using y and multi_x1.
# mlr = sm.OLS(...)

# TODO 2: fit the model.
# model_mlr = ...

# TODO 3: display or print the model summary.
# ...

In [ ]:
# TODO: Generate the predicted SalePrice values for the MLR model.
# mlr_y_pred = ...

# TODO: Compute MLR evaluation metrics.
# mlr_r2 = r2_score(...)
# mlr_mse = mean_squared_error(...)
# mlr_rmse = ...

# TODO: Print the metrics using the SLR cell as a template.
# print(...)

In [ ]:
# Diagnostic plot: residual plot.
# Run this after you have created mlr_y_pred above.

from sklearn.metrics import PredictionErrorDisplay

# TODO: uncomment and run after completing the previous cell.
# display = PredictionErrorDisplay(y_true=y, y_pred=mlr_y_pred)
# display.plot()
# plt.title('MLR: Residual Plot')
# plt.show()

<div style="background-color: #fff899; padding: 10px; border-radius: 5px;">
  <h3 style="color: #0056b3;">Step 5. Reflection / Exit Ticket</h3>
  <p>Answer these questions in 2-4 sentences each:</p>
  <ol>
    <li>Is the MLR model better than the SLR model? Use R^2 and RMSE to justify your answer.</li>
    <li>Which predictor has the strongest evidence of being useful? How can you tell from the summary?</li>
    <li>Look at the actual-vs-predicted plot. What pattern or limitation do you notice?</li>
    <li>What is one way this model could be improved in a future lab?</li>
  </ol>
</div>